In [1]:
from excel_export import DeformationRecorder
import PISO
from viz import plot_history, plot_membrane_3d, plot_membrane_and_slice, save_deformation_gif


['nested_scopes', 'generators', 'division', 'absolute_import', 'with_statement', 'print_function', 'unicode_literals', 'barry_as_FLUFL', 'generator_stop', 'annotations']


In [2]:
import sys
import importlib

# 1. Clear any cached memory of the Coupling module
if "Coupling" in sys.modules:
    del sys.modules["Coupling"]

# 2. Re-import your updated module
import Coupling
from Coupling import QuasiStaticFSI

print("Coupling.py successfully reloaded from disk into Jupyter's memory!")


Coupling.py successfully reloaded from disk into Jupyter's memory!


In [3]:
with open("Coupling.py", "r", encoding="utf-8") as f:
    lines = f.readlines()

# Define the missing data tracking class structure
history_class_lines = [
    "\n",
    "from dataclasses import dataclass, field\n",
    "\n",
    "@dataclass\n",
    "class QuasiStaticHistory:\n",
    "    iteration: list = field(default_factory=list)\n",
    "    max_disp: list = field(default_factory=list)\n",
    "    shape_residual: list = field(default_factory=list)\n",
    "    uwm_residual: list = field(default_factory=list)\n",
    "    pressure_max: list = field(default_factory=list)\n",
    "    cfl: list = field(default_factory=list)\n",
    "\n"
]

# Ensure we don't accidentally double-inject it if it's hiding elsewhere
file_content = "".join(lines)
if "class QuasiStaticHistory" not in file_content:
    # Safely insert it right after the first few lines of documentation / package imports
    lines.insert(5, "".join(history_class_lines))
    with open("Coupling.py", "w", encoding="utf-8") as f:
        f.writelines(lines)
    print("QuasiStaticHistory has been forcefully injected into Coupling.py!")
else:
    print("Class already exists in the file text layout.")


Class already exists in the file text layout.


In [ ]:
"""Entry point: quasi-static UWM ↔ PISO/LES membrane FSI."""

from __future__ import annotations

import argparse
import csv
import sys
from pathlib import Path
import os

import numpy as np
import mesh
import PISO
import FSI_load_transfer

# Standalone package root (independent of tensile_membrane_fsi / repo src)
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path(os.getcwd()).resolve()

QS = ROOT

if str(QS) not in sys.path:
    sys.path.insert(0, str(QS))

from io_helper import load_config, ensure_dir, save_snapshot, write_membrane_xdmf
from viz import (
    plot_history,
    plot_membrane_3d,
    plot_membrane_and_slice,
    save_deformation_gif,
)
from Coupling import QuasiStaticFSI
from excel_export import DeformationRecorder


def parse_args():
    p = argparse.ArgumentParser(
        description="Quasi-static FSI: Updated Weight Method + PISO/LES"
    )
    p.add_argument(
        "-c",
        "--config",
        type=str,
        default=str(QS / "config" / "quasi_static.yaml"),
        help="Path to YAML config",
    )
    p.add_argument(
        "--quick",
        action="store_true",
        help="Coarse mesh / few iterations smoke run",
    )
    p.add_argument(
        "--excel",
        type=str,
        default=None,
        help="Output Excel path (default: <output_dir>/membrane_deformations.xlsx)",
    )

    # Notebook Fix: Parse only the flags defined above, ignore Jupyter's internal flags
    args, unknown = p.parse_known_args()
    return args


def write_deformation_gifs(recorder, sim, out, cfg, fps: int = 8):
    """Write physical + amplified membrane deformation GIFs from recorder history."""
    if not recorder.nodes:
        print("[QS-FSI] no recorded frames — skipping deformation GIF")
        return None, None

    gif_path = Path(out) / "membrane_deformation.gif"
    save_deformation_gif(
        nodes_over_time=recorder.nodes,
        elements=sim.mesh.elements,
        nodes0=recorder.reference,
        out_path=gif_path,
        times=recorder.times,
        fps=fps,
        amplify=1.0,
        title="Quasi-static membrane deformation",
    )
    print(f"[QS-FSI] deformation GIF → {gif_path}")

    amp = float(cfg["simulation"].get("gif_amplify", 5.0))
    gif_amp = None
    if amp != 1.0:
        gif_amp = Path(out) / "membrane_deformation_amplified.gif"
        save_deformation_gif(
            nodes_over_time=recorder.nodes,
            elements=sim.mesh.elements,
            nodes0=recorder.reference,
            out_path=gif_amp,
            times=recorder.times,
            fps=fps,
            amplify=amp,
            title=f"Membrane deformation (×{amp:g})",
        )
        print(f"[QS-FSI] amplified deformation GIF → {gif_amp}")
    return gif_path, gif_amp


def main():
    args = parse_args()
    cfg = load_config(args.config)

    if args.quick:
        # Membrane size/resolution come from geometry.py defaults (not YAML).
        cfg["fluid"]["nx"] = 16
        cfg["fluid"]["ny"] = 8
        cfg["fluid"]["nz"] = 8
        cfg["fluid"]["nu"] = 5.0e-3
        cfg["time"]["dt"] = 0.01
        cfg["quasi_static"]["max_iters"] = 3
        cfg["quasi_static"]["fluid_substeps"] = 5
        cfg["quasi_static"]["load_scale"] = 0.15
        cfg["les"]["enabled"] = True
        cfg["simulation"]["save_interval"] = 1

    out_rel = cfg["simulation"].get("output_dir", "output")
    out = ensure_dir(ROOT / out_rel)
    print(f"[QS-FSI] output → {out}")

    sim = QuasiStaticFSI(cfg)
    print(
        f"[QS-FSI] UWM membrane {sim.mesh.nx}x{sim.mesh.ny}, "
        f"PISO+LES fluid {cfg['fluid']['nx']}x{cfg['fluid']['ny']}x{cfg['fluid']['nz']}, "
        f"max_iters={cfg['quasi_static']['max_iters']}"
    )
    z0 = float(cfg["fluid"]["membrane_z0"])
    nodes_flat = sim.x_bc.copy()
    nodes_flat[:, 2] = z0
    recorder = DeformationRecorder(reference_nodes=nodes_flat, fixed=sim.mesh.fixed)

    def on_iter(simulation, info, k):
        print(
            f"  iter={int(info['iteration']):02d}  "
            f"disp={info['max_disp']:.4e} m  "
            f"shape_res={info['shape_residual']:.2e}  "
            f"uwm_res={info['uwm_residual']:.2e}  "
            f"|p|_max={info['pressure_max']:.2f} Pa  "
            f"CFL={info['cfl']:.3f}"
        )
        recorder.record(
            time=float(info.get("time", info["iteration"])),
            nodes=simulation.nodes,
            iteration=int(info["iteration"]),
        )
        save_every = int(cfg["simulation"].get("save_interval", 1))
        if k % save_every == 0:
            save_snapshot(
                out,
                step=int(info["iteration"]),
                time=float(info["iteration"]),
                membrane_nodes=simulation.nodes,
                membrane_elements=simulation.mesh.elements,
                fluid_u=simulation.fluid.state.cell_centered_velocity()[0],
                fluid_v=simulation.fluid.state.v,
                fluid_w=simulation.fluid.state.w,
                fluid_p=simulation.fluid.state.p,
                meta={
                    "shape_residual": info["shape_residual"],
                    "max_disp": info["max_disp"],
                },
            )
            step = int(info["iteration"])
            nodes = simulation.nodes
            elements = simulation.mesh.elements
            disp = nodes - recorder.reference
            disp_mag = np.linalg.norm(disp, axis=1)
            write_membrane_xdmf(
                out / f"membrane_{step:06d}.xdmf",
                nodes,
                elements,
                point_data={
                    "displacement": disp,
                    "disp_mag": disp_mag,
                },
            )

    hist = sim.run(callback=on_iter)

    # history CSV
    csv_path = out / "history.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(
            [
                "iteration",
                "max_disp",
                "shape_residual",
                "uwm_residual",
                "pressure_max",
                "cfl",
            ]
        )
        for i in range(len(hist.iteration)):
            w.writerow(
                [
                    hist.iteration[i],
                    hist.max_disp[i],
                    hist.shape_residual[i],
                    hist.uwm_residual[i],
                    hist.pressure_max[i],
                    hist.cfl[i],
                ]
            )

    # reuse plot_history with synthetic time = iteration index
    class _H:
        pass

    h = _H()
    h.time = [float(v) for v in hist.iteration]
    h.max_disp = hist.max_disp
    h.kinetic = hist.pressure_max  # show |p|_max in the KE panel slot
    h.cfl = hist.cfl
    h.residual = hist.shape_residual
    if cfg["simulation"].get("plot", True):
        plot_history(h, out / "history.png")
        plot_membrane_3d(
            sim.nodes,
            sim.mesh.elements,
            out / "final_membrane.png",
            title="Quasi-static UWM form under fluid load",
        )
        j = sim.grid.ny // 2
        plot_membrane_and_slice(
            sim.nodes,
            sim.mesh.elements,
            sim.fluid.state.cell_centered_velocity()[0],
            sim.grid.x,
            sim.grid.z,
            j,
            out / "final_slice.png",
            title="Quasi-static UWM + PISO/LES",
        )

    print(f"[QS-FSI] done — {len(hist.iteration)} outer iterations → {out}")
    excel_path = Path(args.excel) if args.excel else out / "membrane_deformations.xlsx"
    xlsx = recorder.write_xlsx(
        excel_path, per_step_sheets=len(recorder.times) <= 40
    )
    print(f"[QS-FSI] deformations Excel → {xlsx}")

    # Animated GIF of membrane deformation from recorded iterations
    if recorder.nodes:
        gif_path = out / "membrane_deformation.gif"
        save_deformation_gif(
            nodes_over_time=recorder.nodes,
            elements=sim.mesh.elements,
            nodes0=recorder.reference,
            out_path=gif_path,
            times=recorder.times,
            fps=8,
            amplify=1.0,
            title="Quasi-static membrane deformation",
        )
        print(f"[QS-FSI] deformation GIF → {gif_path}")

        # Optional amplified view (easier to see small deflections)
        amp = float(cfg["simulation"].get("gif_amplify", 5.0))
        if amp != 1.0:
            gif_amp = out / "membrane_deformation_amplified.gif"
            save_deformation_gif(
                nodes_over_time=recorder.nodes,
                elements=sim.mesh.elements,
                nodes0=recorder.reference,
                out_path=gif_amp,
                times=recorder.times,
                fps=8,
                amplify=amp,
                title=f"Membrane deformation (×{amp:g})",
            )
            print(f"[QS-FSI] amplified deformation GIF → {gif_amp}")


if __name__ == "__main__":
    main()


In [ ]:
# Stage 2 — deformation capture AFTER form-finding (Coupling.run unchanged)
# Run this after the main QS-FSI cell has finished sim.run().
#
# Requires in memory: `sim` (form-found QuasiStaticFSI), and optionally `out`.

from pathlib import Path
from deformation import MembraneDeformationCapture, capture_deformation_after_form_find

assert "sim" in globals(), "Run the form-finding main cell first so `sim` exists."
out_dir = Path(out) if "out" in globals() else Path("output")
out_dir.mkdir(parents=True, exist_ok=True)

def on_deform(simulation, info, k):
    print(
        f"  deform {k:03d}  t={info['time']:.3f}s  "
        f"|u|_form={info['max_disp_from_form']:.4e} m  "
        f"E={info['energy']:.4e}  "
        f"|p|_max={info['pressure_max']:.2f} Pa"
    )

cap = MembraneDeformationCapture(sim, freeze_weights=True)
hist = cap.run(t_end=sim.t_end, callback=on_deform)

gif = cap.write_gif(out_dir / "membrane_deformation.gif", fps=8, amplify=1.0)
gif_amp = cap.write_gif(
    out_dir / "membrane_deformation_amplified.gif",
    fps=8,
    amplify=5.0,
    title="Membrane deformation (×5)",
)
print(f"[deform] frames={len(hist.time)} → {gif}")
print(f"[deform] amplified → {gif_amp}")


In [ ]:
#!/usr/bin/env python3
"""Quasi-static UWM + PISO/LES over a time interval → animated GIFs.

Advances the fluid in time; at each sample the membrane form is updated
with the Updated Weight Method under the current pressure, then frames
are rendered (3D membrane + mid-plane |u| slice).

Writes **two** GIFs by default:
  - ``membrane_quasi_static_original.gif`` — physical (unamplified) deflection
  - ``membrane_quasi_static_amplified.gif`` — visually scaled deflection

    python main.py --quick
    python run_gif.py --quick
    python run_gif.py --t-end 1.0 --fps 8
    python run_gif.py --out-dir output
"""

from __future__ import annotations

import argparse
import csv
import sys
import time as walltime
from pathlib import Path

import numpy as np

# Standalone package root (independent of tensile_membrane_fsi / repo src)
ROOT = Path(__file__).resolve().parent
QS = ROOT
sys.path.insert(0, str(QS))

from utils.io import ensure_dir, load_config, save_snapshot, write_membrane_xdmf
from utils.viz import (
    plot_history,
    plot_membrane_3d,
    plot_membrane_and_slice,
    render_flutter_frame,
    save_gif,
)

from coupling import QuasiStaticFSI
from excel_export import DeformationRecorder


def parse_args():
    p = argparse.ArgumentParser(
        description="Quasi-static membrane FSI → original + amplified GIFs"
    )
    p.add_argument(
        "-c",
        "--config",
        type=str,
        default=str(QS / "config" / "quasi_static.yaml"),
    )
    p.add_argument(
        "--t-end",
        type=float,
        default=None,
        help="Physical end time [s] (default: from config time.t_end)",
    )
    p.add_argument(
        "--fluid-substeps",
        type=int,
        default=None,
        help="PISO steps per outer/GIF frame (smaller → more time steps). "
        "Default: config quasi_static.fluid_substeps",
    )
    p.add_argument("--fps", type=int, default=None, help="GIF frames per second (default: realtime)")
    p.add_argument(
        "--realtime",
        action="store_true",
        default=True,
        help="Play GIF in real time: total length = t_end seconds (default)",
    )
    p.add_argument(
        "--no-realtime",
        action="store_true",
        help="Use --fps instead of real-time frame delay",
    )
    p.add_argument(
        "--out-dir",
        type=str,
        default=None,
        help="Directory for GIF outputs (default: config simulation.output_dir)",
    )
    p.add_argument(
        "--out",
        type=str,
        default=None,
        help="Deprecated alias: if set, amplified GIF path; original is "
        "written beside it as *_original.gif",
    )
    p.add_argument(
        "--quick",
        action="store_true",
        help="Coarse mesh, short interval smoke GIF",
    )
    p.add_argument(
        "--snapshot-every",
        type=int,
        default=5,
        help="Frames between NPZ/XDMF writes (0 disables)",
    )
    p.add_argument(
        "--disp-scale",
        type=float,
        default=None,
        help="Visual amplification of out-of-plane deflection vs the flat "
        "mounting plane (physics unchanged). Default: auto-scale so the "
        "peak |Δz| fills ~40%% of the membrane span, or config disp_scale.",
    )
    p.add_argument(
        "--target-amp",
        type=float,
        default=None,
        help="Target visual peak |Δz| [m] for auto scaling (default: 0.4×span)",
    )
    p.add_argument(
        "--excel",
        type=str,
        default=None,
        help="Output Excel path for nodal x,y,z at each time step "
        "(default: <output_dir>/membrane_deformations.xlsx)",
    )
    p.add_argument(
        "--no-excel",
        action="store_true",
        help="Skip writing the deformations Excel workbook",
    )
    return p.parse_args()


def _amplify_z(
    nodes: np.ndarray,
    z_ref: float,
    scale: float,
) -> np.ndarray:
    """Amplify out-of-plane deflection about the flat mounting plane (GIF only)."""
    out = np.asarray(nodes, dtype=float).copy()
    out[:, 2] = float(z_ref) + float(scale) * (out[:, 2] - float(z_ref))
    return out


def _auto_disp_scale(nodes: np.ndarray, z_ref: float, target_amp: float) -> float:
    phys = float(np.max(np.abs(nodes[:, 2] - z_ref)))
    return float(target_amp) / max(phys, 1e-9)


def _render_frame(
    nodes_plot: np.ndarray,
    elements: np.ndarray,
    nodes_flat: np.ndarray,
    speed: np.ndarray,
    grid_x: np.ndarray,
    grid_z: np.ndarray,
    time: float,
    mesh_nx: int,
    mesh_ny: int,
    speed_max: float,
    disp_max: float,
    z_limits: tuple,
    title: str,
):
    return render_flutter_frame(
        nodes_plot,
        elements,
        nodes_flat,
        speed,
        grid_x,
        grid_z,
        time,
        mesh_nx,
        mesh_ny,
        speed_max,
        disp_max,
        z_limits,
        title=title,
    )


def main():
    args = parse_args()
    cfg = load_config(args.config)

    if args.quick:
        # Membrane size/resolution come from geometry.py defaults (not YAML).
        cfg["fluid"]["nx"] = 16
        cfg["fluid"]["ny"] = 8
        cfg["fluid"]["nz"] = 8
        cfg["fluid"]["nu"] = 5.0e-3
        cfg["time"]["dt"] = 0.01
        cfg["time"]["t_end"] = 1.0
        cfg["quasi_static"]["fluid_substeps"] = 2
        cfg["quasi_static"]["max_iters"] = 200
        cfg["quasi_static"]["load_scale"] = 1.5
        cfg["quasi_static"]["shape_tol"] = 0.0
        cfg["les"]["enabled"] = True

    if args.t_end is not None:
        cfg["time"]["t_end"] = args.t_end
    if args.fluid_substeps is not None:
        cfg["quasi_static"]["fluid_substeps"] = int(args.fluid_substeps)

    cfg.setdefault("quasi_static", {})
    cfg["quasi_static"]["shape_tol"] = 0.0

    out = ensure_dir(
        Path(args.out_dir)
        if args.out_dir
        else ROOT / cfg["simulation"].get("output_dir", "output")
    )
    if args.out:
        amp_gif_path = Path(args.out)
        orig_gif_path = amp_gif_path.with_name(
            amp_gif_path.stem + "_original" + amp_gif_path.suffix
        )
    else:
        orig_gif_path = out / "membrane_quasi_static_original.gif"
        amp_gif_path = out / "membrane_quasi_static_amplified.gif"

    sim = QuasiStaticFSI(cfg)
    z0 = float(cfg["fluid"]["membrane_z0"])
    nodes_flat = sim.x_bc.copy()
    nodes_flat[:, 2] = z0
    mesh_nx, mesh_ny = sim.mesh.nx, sim.mesh.ny
    j_slice = sim.grid.ny // 2
    span = float(sim.mesh.length)
    target_amp = float(
        args.target_amp
        if args.target_amp is not None
        else cfg.get("quasi_static", {}).get("target_amp", 0.4 * span)
    )

    if args.disp_scale is not None:
        disp_scale = float(args.disp_scale)
    elif cfg.get("quasi_static", {}).get("disp_scale", None) not in (None, "auto"):
        disp_scale = float(cfg["quasi_static"]["disp_scale"])
    else:
        disp_scale = _auto_disp_scale(sim.nodes, z0, target_amp)

    U = float(cfg["fluid"]["U_inlet"])
    speed_max = 1.6 * U
    # Physical GIF: tight window around the true membrane motion
    phys_disp_max = max(
        0.02,
        1.5 * float(np.max(np.abs(sim.nodes[:, 2] - z0))) + 1e-3,
    )
    phys_z_span = max(1.5 * phys_disp_max, 0.08)
    phys_z_limits = (z0 - phys_z_span, z0 + phys_z_span)

    # Amplified GIF scales
    amp_disp_max = max(0.15, 1.15 * target_amp)
    amp_z_span = max(1.25 * amp_disp_max, 0.35)
    amp_z_limits = (z0 - amp_z_span, z0 + amp_z_span)

    frames_orig = []
    frames_amp = []
    t0 = walltime.time()
    t_end = float(cfg["time"]["t_end"])
    recorder = DeformationRecorder(reference_nodes=nodes_flat, fixed=sim.mesh.fixed)

    def on_frame(simulation, info, k):
        nonlocal disp_scale, phys_disp_max, phys_z_limits, amp_disp_max, amp_z_limits
        st = simulation.fluid.state
        speed = np.sqrt(
            st.cell_centered_velocity()[0][:, j_slice, :] ** 2
            + st.v[:, j_slice, :] ** 2
            + st.w[:, j_slice, :] ** 2
        )
        if args.disp_scale is None and cfg.get("quasi_static", {}).get(
            "disp_scale", "auto"
        ) in (None, "auto"):
            disp_scale = _auto_disp_scale(simulation.nodes, z0, target_amp)

        phys_peak = float(np.max(np.abs(simulation.nodes[:, 2] - z0)))
        phys_disp_max = max(0.02, 1.5 * phys_peak + 1e-3, phys_disp_max)
        phys_z_span_now = max(1.5 * phys_disp_max, 0.08)
        phys_z_limits = (z0 - phys_z_span_now, z0 + phys_z_span_now)

        nodes_amp = _amplify_z(simulation.nodes, z0, disp_scale)
        vis_peak = float(np.max(np.abs(nodes_amp[:, 2] - z0)))
        amp_disp_max = max(0.15, 1.15 * vis_peak, 1.15 * target_amp)
        amp_z_span_now = max(1.25 * amp_disp_max, 0.35)
        amp_z_limits = (z0 - amp_z_span_now, z0 + amp_z_span_now)

        recorder.record(
            time=float(info["time"]),
            nodes=simulation.nodes,
            iteration=int(info["iteration"]),
        )

        frames_orig.append(
            _render_frame(
                simulation.nodes,
                simulation.mesh.elements,
                nodes_flat,
                speed,
                simulation.grid.x,
                simulation.grid.z,
                info["time"],
                mesh_nx,
                mesh_ny,
                speed_max,
                phys_disp_max,
                phys_z_limits,
                title="Quasi-static UWM membrane  (original Δz)",
            )
        )
        frames_amp.append(
            _render_frame(
                nodes_amp,
                simulation.mesh.elements,
                nodes_flat,
                speed,
                simulation.grid.x,
                simulation.grid.z,
                info["time"],
                mesh_nx,
                mesh_ny,
                speed_max,
                amp_disp_max,
                amp_z_limits,
                title=f"Quasi-static UWM membrane  (×{disp_scale:.0f} Δz vs flat)",
            )
        )
        print(
            f"  frame {len(frames_orig):3d}  t={info['time']:6.3f}s  "
            f"Δz_phys={phys_peak:.4e} m  "
            f"×{disp_scale:.0f} → {vis_peak:.3f} m  "
            f"|p|_max={info['pressure_max']:.2f} Pa  "
            f"[{walltime.time() - t0:6.1f}s wall]"
        )
        if args.snapshot_every > 0 and (len(frames_orig) - 1) % args.snapshot_every == 0:
            save_snapshot(
                out,
                step=int(info["iteration"]),
                time=float(info["time"]),
                membrane_nodes=simulation.nodes,
                membrane_elements=simulation.mesh.elements,
                fluid_u=st.cell_centered_velocity()[0],
                fluid_v=st.v,
                fluid_w=st.w,
                fluid_p=st.p,
                meta=info,
            )
            step = int(info["iteration"])
            nodes = simulation.nodes
            elements = simulation.mesh.elements
            disp = nodes - recorder.reference
            disp_mag = np.linalg.norm(disp, axis=1)
            write_membrane_xdmf(
                out / f"membrane_{step:06d}.xdmf",
                nodes,
                elements,
                point_data={
                    "displacement": disp,
                    "disp_mag": disp_mag,
                },
            )
            plot_membrane_and_slice(
                simulation.nodes,
                simulation.mesh.elements,
                st.cell_centered_velocity()[0],
                simulation.grid.x,
                simulation.grid.z,
                j_slice,
                out / f"slice_{int(info['iteration']):06d}.png",
                title=f"Quasi-static UWM  t={info['time']:.3f}s",
            )

    print(
        f"[QS-GIF] fluid {sim.grid.nx}x{sim.grid.ny}x{sim.grid.nz}, "
        f"membrane {mesh_nx}x{mesh_ny}, dt={sim.dt}, "
        f"substeps={sim.fluid_substeps}, t_end={t_end}"
    )
    hist = sim.run_timed(t_end=t_end, callback=on_frame)

    csv_path = out / "history.csv"
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(
            [
                "iteration",
                "time",
                "max_disp",
                "shape_residual",
                "uwm_residual",
                "pressure_max",
                "cfl",
            ]
        )
        dt_block = sim.fluid_substeps * sim.dt
        for i in range(len(hist.iteration)):
            w.writerow(
                [
                    hist.iteration[i],
                    hist.iteration[i] * dt_block,
                    hist.max_disp[i],
                    hist.shape_residual[i],
                    hist.uwm_residual[i],
                    hist.pressure_max[i],
                    hist.cfl[i],
                ]
            )

    class _H:
        pass

    h = _H()
    h.time = [i * dt_block for i in hist.iteration]
    h.max_disp = hist.max_disp
    h.kinetic = hist.pressure_max
    h.cfl = hist.cfl
    h.residual = hist.shape_residual
    plot_history(h, out / "history.png")
    plot_membrane_3d(
        sim.nodes,
        sim.mesh.elements,
        out / "membrane_final.png",
        displacement=sim.nodes - nodes_flat,
        title="Final quasi-static UWM form",
    )

    if not frames_orig:
        raise SystemExit("no frames recorded — check t_end / fluid_substeps")

    dt_block = sim.fluid_substeps * sim.dt
    use_realtime = not args.no_realtime and args.fps is None
    if use_realtime:
        # one frame per outer step spanning dt_block of physical time
        duration_ms = max(int(round(1000.0 * dt_block)), 20)
        fps_eff = 1000.0 / duration_ms
        save_gif(frames_orig, orig_gif_path, duration_ms=duration_ms)
        save_gif(frames_amp, amp_gif_path, duration_ms=duration_ms)
        play_s = len(frames_orig) * duration_ms / 1000.0
        print(
            f"[QS-GIF] realtime playback ≈ {play_s:.2f}s "
            f"({len(frames_orig)} frames × {duration_ms} ms, ~{fps_eff:.1f} fps)"
        )
    else:
        fps = int(args.fps) if args.fps is not None else 8
        save_gif(frames_orig, orig_gif_path, fps=fps)
        save_gif(frames_amp, amp_gif_path, fps=fps)
        print(f"[QS-GIF] playback at {fps} fps")

    print(f"[QS-GIF] {len(frames_orig)} frames → {orig_gif_path}  (original)")
    print(f"[QS-GIF] {len(frames_amp)} frames → {amp_gif_path}  (amplified)")
    print(
        f"[QS-GIF] physical time: t = 0 → {hist.iteration[-1] * dt_block:.3f} s "
        f"(last frame labeled t = {sim.time:.2f} s)"
    )
    # keep legacy filename as a copy of the amplified GIF for older docs/links
    legacy = out / "membrane_quasi_static.gif"
    if amp_gif_path.resolve() != legacy.resolve():
        try:
            legacy.write_bytes(amp_gif_path.read_bytes())
        except OSError:
            pass

    if not args.no_excel:
        excel_path = (
            Path(args.excel)
            if args.excel
            else out / "membrane_deformations.xlsx"
        )
        per_step = len(recorder.times) <= 40
        xlsx = recorder.write_xlsx(excel_path, per_step_sheets=per_step)
        print(
            f"[QS-GIF] deformations Excel ({len(recorder.times)} steps × "
            f"{sim.mesh.n_nodes} nodes) → {xlsx}"
        )
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [2]:
import inspect
from PISO import FluidSolver
print(inspect.signature(FluidSolver.__init__))


(self, grid: 'FluidGrid', rho: 'float' = 1.225, nu: 'float' = 1.5e-05, U_inlet: 'float' = 10.0, use_les: 'bool' = True, Cs: 'float' = 0.17, u_clip: 'Optional[float]' = None, gust_amp: 'float' = 0.0, gust_freq: 'float' = 1.0, n_correctores: 'int' = 2) -> 'None'
